# Control continuu cu PPO în MuJoCo

Antrenăm robotul HalfCheetah să alerge în simulatorul fizic MuJoCo, cu algoritmul PPO (Proximal Policy Optimization), implementat în PyTorch.

Notebookul rulează scripturile laboratorului [Control continuu cu PPO în MuJoCo](https://inteligentartificiala.ro/laboratoare/control-continuu-ppo-halfcheetah/prezentare/), de pe [InteligentArtificiala.ro](https://inteligentartificiala.ro), unde este explicat codul fiecărui fișier. Simularea fizică rulează pe procesor, iar mașinile virtuale Colab au puține nuclee, așa că antrenarea de 1.000.000 de pași durează aici mai mult decât pe un calculator cu 4–8 nuclee.

Placa grafică este necesară pentru videoclipul de la final: fără ecran, MuJoCo desenează cadrele pe placa grafică. Dacă nu este deja aleasă, se alege din meniul **Runtime**, opțiunea **Change runtime type**, o variantă cu GPU. Celulele se rulează în ordine, de sus în jos.

## Codul laboratorului

Fișierele se descarcă de pe site, ca în varianta cu `curl` de pe [pagina de introducere](https://inteligentartificiala.ro/laboratoare/control-continuu-ppo-halfcheetah/prezentare/#pregătirea-mediului-de-lucru). Pe lângă pachetele din `requirements.txt`, înregistrarea videoclipului cu `RecordVideo` din Gymnasium are nevoie de `moviepy`.

In [ ]:
%%bash
mkdir -p ppo-halfcheetah && cd ppo-halfcheetah
for f in requirements.txt model.py buffer.py train.py evaluate.py; do
  curl -fsSO "https://inteligentartificiala.ro/laboratoare/control-continuu-ppo-halfcheetah/cod/$f"
done
ls

In [ ]:
%cd ppo-halfcheetah
%pip install -q -r requirements.txt moviepy

## Desenarea cadrelor fără ecran

MuJoCo desenează cadrele videoclipului prin EGL, direct pe placa grafică. Celula de mai jos face driverul NVIDIA vizibil pentru EGL, la fel ca în notebookul oficial MuJoCo, și alege acest mod. Scripturile rulate cu `!` moștenesc variabila de mediu `MUJOCO_GL`.

In [ ]:
import os

NVIDIA_ICD = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(NVIDIA_ICD):
    with open(NVIDIA_ICD, "w") as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')
os.environ["MUJOCO_GL"] = "egl"

## [Antrenarea](https://inteligentartificiala.ro/laboratoare/control-continuu-ppo-halfcheetah/antrenare-si-evaluare/)

Graficele antrenării se pot urmări în TensorBoard. Celula următoare îl pornește, iar graficele se actualizează în timp ce agentul se antrenează.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
!python train.py

## [Evaluarea și videoclipul](https://inteligentartificiala.ro/laboratoare/control-continuu-ppo-halfcheetah/antrenare-si-evaluare/#evaluarea-și-generarea-videoclipului)

Strategia antrenată rulează determinist 5 episoade, iar fiecare episod este înregistrat într-un videoclip.

In [ ]:
!python evaluate.py --checkpoint checkpoints/ppo_halfcheetah.pt --episoade 5 --video

In [ ]:
import glob

from IPython.display import Video

Video(sorted(glob.glob("videoclipuri/*.mp4"))[0], embed=True, width=480)